# Mini-LLM para a HP 50g

Treino de um modelo de linguagem minúsculo (~1,2M de parâmetros, pesos int4 ou int8) que escreve **diálogos curtos**
em português, para rodar em C (HPGCC) na HP 50g. Tudo fica em `Meu Drive/LLM_Laika/hp50g/`. Detalhes em
`hp50g/LEIA-ME.md`.

Ordem: 1 → 2 (sem GPU) → 3 e 4 (varreduras, L4) → 5 (opcional) → 6 (treino final) → 7 (exportar e relatório).
Um modelo tão pequeno usa mal a A100: a **L4** gasta bem menos unidades e deve ser quase tão rápida aqui.

In [ ]:
# 1) PREPARAR: GPU, Google Drive, código e atalhos para os dados da Laika
RAMO = "main"  #@param {type:"string"}
import os, subprocess, codecs, json, glob
def rodar(comando):
    """Roda um comando mostrando a saída aos poucos; se ele falhar, a célula para com erro."""
    print("$", comando, flush=True)
    p = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env=dict(os.environ, PYTHONUNBUFFERED="1"))
    t = codecs.getincrementaldecoder("utf-8")("replace")
    while pedaco := p.stdout.read1(65536):
        print(t.decode(pedaco), end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"o comando falhou (código {p.returncode}): {comando}")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (só processador)")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"
HP = f"{DRIVE}/hp50g"
os.makedirs(HP, exist_ok=True)
if not os.path.exists("/content/LLM_Laika"):
    rodar("git clone -q https://github.com/BrunoDevPro/LLM_Laika.git /content/LLM_Laika")
os.chdir("/content/LLM_Laika")
rodar(f"git fetch -q origin {RAMO} && git checkout -q -f -B {RAMO} FETCH_HEAD")
rodar("pip install -q tokenizers")
# Os dados que a Laika já baixou (célula 2 do colab.ipynb): Wikipédia, GigaVerbo, Carolina e o SFT.
for pasta in ["wiki", "gigaverbo", "carolina", "sft"]:
    local, remoto = f"data/{pasta}", f"{DRIVE}/{pasta}"
    if os.path.isdir(remoto) and not os.path.islink(local):
        rodar(f"rm -rf {local}")
        os.symlink(remoto, local)
    print(pasta, "->", len(os.listdir(remoto)) if os.path.isdir(remoto) else "não baixado")
LOCAL = "/content/hp50g_dados"  # cópia local dos .bin (ler do Drive durante o treino é lento)
def dados_locais():
    rodar(f"mkdir -p {LOCAL} && rsync -a --include='*.bin' --include='*.json' --exclude='*' {HP}/dados/ {LOCAL}/")
    return LOCAL

In [ ]:
# 2) DADOS E TOKENIZADOR (sem GPU): diálogos curtos + frases simples, BPE de 1024, tokens em binário
# Diálogos: os exemplos escritos da Laika (3×) e as conversas curtas do GigaVerbo-v2-SFT (célula 2 da Laika, SFT).
# Texto simples: frases de 4 a 25 palavras da Wikipédia, do GigaVerbo e do Carolina, em parágrafos de até 5 frases.
# TEXTO_MB: ~3 caracteres por token, então 2000 MB ≈ 650 milhões de tokens. Fica em Drive/LLM_Laika/hp50g/dados.
TEXTO_MB = 2000  #@param {type:"integer"}
SFT_MAXIMO = 0  #@param {type:"integer"}
rodar(f"python hp50g/preparar.py --saida {HP}/dados --texto-mb {TEXTO_MB} --sft-max {SFT_MAXIMO}")

In [ ]:
# 3) VARREDURA CURTA: taxa de aprendizado, SwiGLU × ReLU², variantes A e B (L4 basta)
# Cada combinação treina TOKENS e grava em Drive/LLM_Laika/hp50g/varredura/<nome>. Já feitas são puladas.
TOKENS = "50 milhões"  #@param ["20 milhões", "50 milhões", "100 milhões", "200 milhões"]
VARIANTES = "A,A_relu2,B,B_relu2"  #@param {type:"string"}
TAXAS = "1e-3,3e-3,6e-3"  #@param {type:"string"}
n = {"20 milhões": 2e7, "50 milhões": 5e7, "100 milhões": 1e8, "200 milhões": 2e8}[TOKENS]
dl = dados_locais()
for v in VARIANTES.split(","):
    for lr in TAXAS.split(","):
        saida = f"{HP}/varredura/{v.strip()}_lr{lr.strip()}_{int(n / 1e6)}M"
        if os.path.exists(f"{saida}/resumo.json"):
            continue
        rodar(f"python hp50g/treino.py --variante {v.strip()} --lr {lr.strip()} --tokens {n:.0f} --dados {dl} "
              f"--saida {saida} --compile")
print("| execução | perda float | perda quantizada |\n|---|---|---|")
for r in sorted(glob.glob(f"{HP}/varredura/*/resumo.json")):
    j = json.load(open(r))
    print(f"| {r.split('/')[-2]} | {j['melhor_perda_float']:.4f} | {j['melhor_perda_quant']:.4f} |")

In [ ]:
# 4) VARREDURA DE DADOS: onde a perda quantizada para de cair (200M, 500M, 1B tokens)
VARIANTE = "A"  #@param {type:"string"}
TAXA = 3e-3  #@param {type:"number"}
QUANTIDADES = "200,500,1000"  #@param {type:"string"}
dl = dados_locais()
for m in QUANTIDADES.split(","):
    saida = f"{HP}/dados_x/{VARIANTE}_{m.strip()}M"
    if not os.path.exists(f"{saida}/resumo.json"):
        rodar(f"python hp50g/treino.py --variante {VARIANTE} --lr {TAXA} --tokens {float(m) * 1e6:.0f} --dados {dl} "
              f"--saida {saida} --compile")
for r in sorted(glob.glob(f"{HP}/dados_x/*/resumo.json")):
    j = json.load(open(r))
    print(f"{r.split('/')[-2]}: perda quantizada {j['melhor_perda_quant']:.4f} (float {j['melhor_perda_float']:.4f})")

In [ ]:
# 5) PROFESSOR (opcional): ~23M de parâmetros, mesmo tokenizador, para a destilação na célula 6
TOKENS_PROFESSOR = 3e9  #@param {type:"number"}
dl = dados_locais()
rodar(f"python hp50g/treino.py --variante professor --lr 1e-3 --tokens {TOKENS_PROFESSOR:.0f} --dados {dl} "
      f"--saida {HP}/professor --sem-qat --compile")

In [ ]:
# 6) TREINO FINAL: ~1 bilhão de tokens, QAT na fase de queda (últimos 20%); continua sozinho se cair
VARIANTE = "A"  #@param {type:"string"}
TAXA = 3e-3  #@param {type:"number"}
TOKENS = 1e9  #@param {type:"number"}
DESTILAR = False  #@param {type:"boolean"}
NOME = ""  #@param {type:"string"}
dl = dados_locais()
nome = NOME.strip() or f"{VARIANTE}_final" + ("_dest" if DESTILAR else "")
extra = f" --professor {HP}/professor/melhor_float.pt --alfa 0.5" if DESTILAR else ""
rodar(f"python hp50g/treino.py --variante {VARIANTE} --lr {TAXA} --tokens {TOKENS:.0f} --dados {dl} "
      f"--saida {HP}/final/{nome} --compile{extra}")

In [ ]:
# 7) EXPORTAR E RELATÓRIO: simulador só com inteiros, hplm.bin + vocab.bin + quant.json, tabela e amostras
EXECUCOES = ""  #@param {type:"string"}
# Vazio = todos os treinos finais. Ou nomes separados por vírgula (pastas de hp50g/final ou hp50g/varredura).
JANELAS = 200  #@param {type:"integer"}
nomes = [n.strip() for n in EXECUCOES.split(",") if n.strip()] or sorted(os.listdir(f"{HP}/final"))
pacotes = []
for n in nomes:
    pasta = next(p for p in (f"{HP}/final/{n}", f"{HP}/varredura/{n}", f"{HP}/dados_x/{n}") if os.path.isdir(p))
    rodar(f"python hp50g/simulador.py exportar --ckpt {pasta}/melhor.pt --dados {HP}/dados --saida {HP}/export/{n} "
          f"--nome {n} --janelas {JANELAS}")
    pacotes.append(f"{HP}/export/{n}")
rodar(f"python hp50g/simulador.py relatorio --pacotes {' '.join(pacotes)} --saida {HP}/export/relatorio.md")